### Generate Vectors at each level of an iteration (vector when considering delta # neighbors,then the next delta neighbors and on..)

Each word will end up having as many vectors as the amount of delta neighbors considered

Update word variable to get specific words for comparison later



In [5]:
import contextlib
import numpy as np
import json
import os
os.makedirs("data/indiv_word_representations", exist_ok=True)
os.makedirs("../plots/pca", exist_ok=True)


words = ['king', 'queen', 'man', 'woman', 'long', 'man', 'say', 'water']
delt = [-4, -3, -2, -1, 1, 2, 3, 4]

In [6]:
with open('data/fairytales_word_tf-idfs.json', 'r') as f:
    tf_idfs = json.load(f)
with open('data/fairytales_word_bloom-filters.json', 'r') as f:
    bloom_filters = json.load(f)
with open('data/fairytales_tokenized.json', 'r') as f:
    tokenized_corpus = json.load(f)
with open('data/iterative_vectors/window_4_iter_200.json', 'r') as f:
    iterative_vectors = json.load(f)

In [7]:
for word in bloom_filters.keys(): # rescaling
    bloom_filters[word] = np.array(bloom_filters[word], dtype=int) * (32/3) - 1

### Functions to transform words to each individual vectors per neighbor set

In [8]:
def generate_vector_from_bloom(word, tokenized_sentence, bits, deltas):
    representations = []
    instance_representation = np.zeros(bits)
    indices = [i for i, x in enumerate(tokenized_sentence) if x == word]
    
    for index in indices:
        adjacent_words = 0
        for delta in deltas:
            if index + delta < 0:
                continue
            with contextlib.suppress(IndexError):
                adjacent_word = tokenized_sentence[index + delta]
                try:
                    tf_idf = tf_idfs[word][adjacent_word]
                except KeyError:
                    tf_idf = 0
                instance_representation += np.array(bloom_filters[adjacent_word]) * tf_idf
                adjacent_words += 1
        representations.append((instance_representation/adjacent_words).tolist())
    return representations



# def generate_vector_from_bloom_no_tfidfs(word, tokenized_sentence, bits, deltas):
#     representations = []
#     instance_representation = np.zeros(bits)
#     indices = [i for i, x in enumerate(tokenized_sentence) if x == word]
    
#     for index in indices:
#         adjacent_words = 0
#         for delta in deltas:
#             if index + delta < 0:
#                 continue
#             with contextlib.suppress(IndexError):
#                 adjacent_word = tokenized_sentence[index + delta]
#                 instance_representation += np.array(bloom_filters[adjacent_word])
#                 adjacent_words += 1
#         representations.append((instance_representation/adjacent_words).tolist())
#     return representations

In [9]:
def generate_vector_from_iterative_vectors(word, tokenized_sentence, bits, deltas):
    representations = []
    instance_representation = np.zeros(bits)
    indices = [i for i, x in enumerate(tokenized_sentence) if x == word]
    
    for index in indices:
        adjacent_words = 0
        for delta in deltas:
            if index + delta < 0:
                continue
            with contextlib.suppress(IndexError):
                adjacent_word = tokenized_sentence[index + delta]
                if adjacent_word not in iterative_vectors:
                    continue
                try:
                    tf_idf = tf_idfs[word][adjacent_word]
                except KeyError:
                    tf_idf = 0
                instance_representation += np.array(iterative_vectors[adjacent_word]) * tf_idf
                adjacent_words += 1
        if adjacent_words > 0:
            representations.append((instance_representation/adjacent_words).tolist())
    return representations

In [10]:
def extract_vectors(word, generation_function, deltas=None, bits=32):
    if deltas is None:
        deltas = delt

    representations = []
    for sentence in tokenized_corpus:
        if word in sentence:
            representations += generation_function(word, sentence, bits, deltas)
    return representations

In [11]:
def store_encoding(word, fname, args):
    vector = extract_vectors(word, **args)
    
    with open(fname, 'r') as f:
        vectors = json.load(f)
    vectors[word] = vector
    with open(fname, 'w') as f:
        json.dump(vectors, f, indent=4)

### 

In [12]:
if not os.path.exists('data/indiv_word_representations/generate_vector_from_bloom.json'):
    with open('data/indiv_word_representations/generate_vector_from_bloom.json', 'w') as f:
        f.write('{}')

if not os.path.exists('data/indiv_word_representations/generate_vector_from_iterative_vectors.json'):
    with open('data/indiv_word_representations/generate_vector_from_iterative_vectors.json', 'w') as f:
        f.write('{}')



for word in words:
    # store_encoding(word, 'data/indiv_word_representations/generate_vector_from_bloom.json', 
    #                {'deltas': delt, 'bits':32, 'generation_function':generate_vector_from_bloom})
    store_encoding(word, 'data/indiv_word_representations/generate_vector_from_iterative_vectors.json', 
                   {'deltas': delt, 'bits':32, 'generation_function':generate_vector_from_iterative_vectors})

KeyError: 'fixed'